# 04 Regression
Monthly revenue forecasting benchmark across linear regression, ridge, lasso, elastic net, decision tree, random forest, and SVR.

In [ ]:
%matplotlib inline
from pathlib import Path

from scipy.stats import skew

from src.data_loader import load_telco_data, split_train_val_test
from src.evaluation import plot_actual_vs_predicted, plot_residual_distribution
from src.preprocessing import prepare_train_val_test_features
from src.regressors import benchmark_regressors, extract_linear_coefficients
from src.utils import PROJECT_ROOT, ensure_directory, set_random_seed

set_random_seed(42)
figures_dir = ensure_directory(PROJECT_ROOT / 'reports' / 'figures')
models_dir = ensure_directory(PROJECT_ROOT / 'models')

frame = load_telco_data()
train_frame, validation_frame, test_frame = split_train_val_test(frame, target_column='MonthlyCharges', stratify=False, random_state=42)
X_train, y_train, X_validation, y_validation, X_test, y_test, preprocessor, feature_names = prepare_train_val_test_features(
    train_frame,
    validation_frame,
    test_frame,
    target_column='MonthlyCharges',
    drop_columns=['customerID', 'Churn'],
    scaler='none',
)

comparison_frame, best_regressor, best_row, details = benchmark_regressors(
    X_train,
    y_train,
    X_validation,
    y_validation,
    X_test,
    y_test,
    random_state=42,
    save_path=models_dir / 'best_regressor.pkl',
    return_details=True,
)

print(comparison_frame[['model', 'validation_rmse', 'validation_r2', 'test_mae', 'test_rmse', 'test_r2', 'training_time']])
print('Best model on validation set:', best_row['model'])

detail_lookup = {item['model']: item for item in details}
for item in details:
    model_slug = item['model'].lower().replace(' ', '_')
    plot_actual_vs_predicted(
        y_test,
        item['test_predictions'],
        f'{item[model]} - Actual vs Predicted',
        figures_dir / f'regression_{model_slug}_actual_vs_predicted.png',
    )

best_detail = detail_lookup[best_row['model']]
residuals = y_test - best_detail['test_predictions']
plot_residual_distribution(residuals, f'{best_row[model]} - Residual Distribution', figures_dir / 'regression_best_residuals.png')
print('Residual mean:', round(float(residuals.mean()), 4))
print('Residual skewness:', round(float(skew(residuals)), 4))

for linear_name in ['Linear Regression', 'Ridge', 'Lasso', 'ElasticNet']:
    linear_detail = detail_lookup[linear_name]
    try:
        coefficients = extract_linear_coefficients(linear_detail['estimator'], feature_names)
        print('')
        print(f'{linear_name} top coefficients:')
        print(coefficients.head(10))
    except ValueError:
        print(f'{linear_name} coefficients are unavailable.')

print('Best regressor test metrics:', best_detail['test_metrics'])